# Postprocessing: load bilby results, remake corner plots and summary tables

Run this after `src/run_pe.py` has produced `outdir_<event>_<waveform>/` for all three waveforms of a given event.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import bilby
import pandas as pd
from remnant_fits import compute_remnant_properties, summarize, STANDARD_SUMMARY_KEYS

EVENT = 'GW231002'
WAVEFORMS = ['IMRPhenomXPHM', 'IMRPhenomTPHM', 'IMRPhenomPv3']

results = {
    wf: bilby.result.read_in_result(
        filename=f'../outdir_{EVENT}_{wf}/{EVENT}_result.json')
    for wf in WAVEFORMS
}

In [ ]:
# Overlaid corner plot across all three waveforms (like thesis Fig. 5.1)
fig = bilby.result.plot_multiple(
    list(results.values()),
    labels=list(results.keys()),
    parameters=['chirp_mass', 'mass_ratio', 'chi_eff', 'a_1', 'a_2',
                'mass_1', 'mass_2', 'total_mass'],
)
fig.savefig(f'../results/{EVENT}/corner_intrinsic_overlay.png')

In [ ]:
# Build the summary table (median, +hi, -lo) across waveforms
rows = []
for key in STANDARD_SUMMARY_KEYS + ['final_mass_approx_source', 'final_spin_approx', 'Radiated_energy']:
    row = {'variable': key}
    for wf, res in results.items():
        post = compute_remnant_properties(res.posterior.copy())
        if key in post.columns:
            lo, mid, hi = summarize(post[key], key, sigma=2)
            row[f'{wf}_median'] = mid
            row[f'{wf}_plus'] = hi - mid
            row[f'{wf}_minus'] = mid - lo
    rows.append(row)

pd.DataFrame(rows).to_csv(f'../results/{EVENT}/summary_table_regenerated.csv', index=False)